# MODULE 4: FORMAL STATISTICAL & DIAGNOSTIC ANALYSIS
## Econometric Modeling, Threshold Detection, and Operational Accountability
**Project:** Gradient Learnings Data Analytics Hackathon 2026 — Olist Brazilian E-Commerce Marketplace Diagnostic  
**Lead Data Scientist & Principal Statistician:** Lead Analytical Team  
**Dataset Grain:** 1 Row = 1 Order (`order_id`) | Canonical Model: `data/processed/analytical_model.parquet`  
**Evaluation Standard:** Independent, Zero-Trust, Peer-Defensible Competition Submission  

---
### Analytical Mandate
Moving from **"What is happening?"** (EDA) to:
> **"How strongly is it associated, where does it happen, what explains it, and how robust is the evidence?"**

This notebook establishes the formal empirical bridge between exploratory data analysis and root-cause accountability. Every section follows:
`Question → Method → Result → Interpretation → Business Implication`


## 1. Research & Statistical Methodology
Before executing any models, we establish explicit econometric standards:
* **Binary Logistic Regression:** Modeling low customer review probability ($P(Y \le 2\star)$) using maximum likelihood estimation.
* **Odds Ratios (OR) & 95% Wald CIs:** Reporting multiplicative odds increments alongside standardized continuous steps ($\Delta = 5	ext{d}$ for delay, $\Delta = 500	ext{km}$ for distance).
* **Segmented Piecewise Regression:** Detecting non-linear operational breakpoints via continuous profile likelihood grid search.
* **Variance Inflation Factor (VIF):** Ensuring $	ext{VIF} < 5.0$ to eliminate multicollinearity distortions.
* **Observational Causal Governance:** Distinguishing adjusted associations from counterfactual proof.
Detailed methodology is documented in `research/module_4_statistical_methodology.md`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns

# Set paths
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
TABLES_DIR = BASE_DIR / "outputs" / "tables"
FIGURES_DIR = BASE_DIR / "outputs" / "figures"

print("Statistical Environment & Dependencies Loaded.")
print(f"Working Directory: {BASE_DIR}")


## 2. Analytical Population Definitions
To prevent denominator contamination and ensure full reproducibility, 5 nested populations are strictly governed:
* **Population A ($N = 99,441$):** Complete canonical marketplace base.
* **Population B ($N = 96,478$):** Delivered orders (`order_status == 'delivered'`).
* **Population C ($N = 96,470$):** Eligible delivery orders (delivered with non-null customer delivery date).
* **Population D ($N = 98,673$):** Reviewed orders (`has_review == True`).
* **Population E ($N = 95,824$):** Primary Inferential Sample (Delivered & Reviewed).


In [ ]:
# Load canonical analytical model
df = pd.read_parquet(PROCESSED_DIR / "analytical_model.parquet")

# Verify standardized population denominators
pop_a = len(df)
pop_b = int((df['order_status'] == 'delivered').sum())
pop_c = int(((df['order_status'] == 'delivered') & df['order_delivered_customer_date'].notna()).sum())
pop_d = int(df['has_review'].sum())
pop_e = int(((df['order_status'] == 'delivered') & df['order_delivered_customer_date'].notna() & df['has_review']).sum())

pops_df = pd.DataFrame([
    {"Population": "Population A (All Orders)", "Order_Count": pop_a, "Share_Pct": 100.0, "Role": "Macro Marketplace Census"},
    {"Population": "Population B (Delivered Orders)", "Order_Count": pop_b, "Share_Pct": pop_b / pop_a * 100, "Role": "Funnel Fulfillment Scope"},
    {"Population": "Population C (Eligible Delivery)", "Order_Count": pop_c, "Share_Pct": pop_c / pop_a * 100, "Role": "Logistics Duration Base"},
    {"Population": "Population D (Reviewed Orders)", "Order_Count": pop_d, "Share_Pct": pop_d / pop_a * 100, "Role": "Satisfaction Sensor Base"},
    {"Population": "Population E (Delivered & Reviewed)", "Order_Count": pop_e, "Share_Pct": pop_e / pop_a * 100, "Role": "Primary Inferential Modeling Sample"}
])
display(pops_df)


## 3. Feature Governance & Multicollinearity Audit
Predictors are audited prior to modeling to prevent target leakage, endogenous feedback loop contamination, and severe variance inflation.


In [ ]:
feature_audit = pd.read_csv(TABLES_DIR / "module_4_feature_audit.csv")
display(feature_audit)


## 4. Delivery Delay Continuous Modeling & Threshold Analysis
* **Business Question:** At what specific late delivery threshold does customer dissatisfaction accelerate non-linearly?
* **Statistical Method:** Piecewise segmented regression grid search over candidate breakpoints $	au \in [0.5, 10.0]	ext{ days}$, minimizing Residual Sum of Squares (RSS) and AIC, paired with non-parametric LOWESS smoothing.
* **Findings:** The optimal empirical inflection breakpoint occurs at **$	au = 0.5	ext{ to }3.5	ext{ days late}$**. Beyond Day 3, the slope of star rating collapse accelerates from $-0.018\star/	ext{day}$ to $-0.068\star/	ext{day}$.
* **Business Implication:** Defines the operational SLA emergency threshold: packages delayed past 3 days require immediate proactive customer intervention.


In [ ]:
threshold_df = pd.read_csv(TABLES_DIR / "delay_threshold_analysis.csv")
print("Top 5 Candidate Breakpoints by RSS Reduction:")
display(threshold_df.head(5))

# Display Publication Figures 19 & 20
from IPython.display import Image, display
display(Image(filename=str(FIGURES_DIR / "fig19_delay_vs_predicted_low_review_probability.png")))
display(Image(filename=str(FIGURES_DIR / "fig20_delay_threshold_piecewise_spline_fit.png")))


## 5. Progressive Nested Logistic Regressions (Models 1–5)
* **Business Question:** Does delivery delay retain its predictive power after progressively controlling for transit duration, spatial distance, commercial order value, regional geography, product category, and feedback timing?
* **Method:** Nested hierarchical logistic regression modeling $P(	ext{low\_review} \le 2\star)$.
* **Findings:** Delivery delay remains overwhelmingly significant across all 5 models ($p < 10^{-50}$). Every 5 days of delay increases the odds of a low review by $+5.6\%$ in continuous linear terms, while categorical severity buckets show a $9.8	ext{x}$ increase for $4–7$ days late.


In [ ]:
logistic_models_df = pd.read_csv(TABLES_DIR / "module_4_logistic_models.csv")
display(logistic_models_df[['model_name', 'predictor', 'odds_ratio', 'ci_lower_95', 'ci_upper_95', 'p_value', 'mcfadden_pseudo_r2']])

# Multicollinearity Check
vif_df = pd.read_csv(TABLES_DIR / "module_4_vif.csv")
print("
Multicollinearity VIF Diagnostic Table:")
display(vif_df)


## 6. Signature Discovery: Survey Timing Asynchrony
* **Business Question:** Does surveying delayed customers before delivery independently amplify negative review scores, or is it merely a proxy for severe shipping delays?
* **Method:** Controlled stratified regression holding delay severity constant, testing 4 timing definitions (A–D).
* **Findings:** Pre-delivery surveys exhibit an independent **Odds Ratio of $4.43$ ($95\%	ext{ CI: }[4.18, 4.69], p < 10^{-15}$)** after controlling for delay days, shipping duration, and order value.
* **Reframed Interpretation:** Fulfillment-SLA Asynchrony: when shipments are delayed, the automated survey fires upon promise date expiration, soliciting ratings during peak customer anxiety before the parcel is delivered.


In [ ]:
st_defs = pd.read_csv(TABLES_DIR / "survey_timing_definitions.csv")
print("Survey Timing Definition Sensitivities (Definitions A-D):")
display(st_defs)

st_models = pd.read_csv(TABLES_DIR / "survey_timing_models.csv")
print("
Nested Survey Timing Models:")
display(st_models)

st_strat = pd.read_csv(TABLES_DIR / "survey_timing_controlled_comparison.csv")
print("
Delay-Stratified Controlled Comparison (Holding Delay Constant):")
display(st_strat)

display(Image(filename=str(FIGURES_DIR / "fig21_survey_timing_adjusted_odds_comparison.png")))


## 7. Delivery Accountability: Seller Handling vs. Carrier Transit
* **Business Question:** Where does fulfillment breakdown occur: in merchant warehouse dispatch or in 3PL carrier transit?
* **Method:** Decomposing fulfillment timeline into Seller Handling (`approval_to_carrier_days`) vs. Carrier Transit (`carrier_to_delivery_days`) and fitting standardized z-score logistic models.
* **Findings:** Carrier transit duration accounts for **$82.5\%$** of total fulfillment duration ($12.1	ext{ days}$ carrier vs. $2.8	ext{ days}$ seller). In standardized terms, carrier transit delay exerts an **Odds Ratio of $1.48$ per SD vs. $1.12$ per SD** for seller handling.
* **Business Implication:** Olist executive leadership must focus operational reforms on 3PL carrier contracts and regional hubs rather than merchant fulfillment penalties.


In [ ]:
decomp_df = pd.read_csv(TABLES_DIR / "delivery_component_models.csv")
display(decomp_df)

display(Image(filename=str(FIGURES_DIR / "fig22_delivery_accountability_seller_vs_carrier.png")))


## 8. Geographic Controls & High-Volume Corridor Risk Matrix
* **Business Question:** Is geographic distance a direct driver of dissatisfaction, or does it operate entirely through transit duration?
* **Method:** Comparing spatial specifications (distance only, state fixed effects, and volume-filtered macro corridors with $N \ge 100$).
* **Findings:** Once carrier transit duration is conditioned out, spatial distance has minimal direct negative impact. Geography operates as a structural bottleneck creating transit delay.


In [ ]:
geo_models_df = pd.read_csv(TABLES_DIR / "geographic_model_comparison.csv")
display(geo_models_df)

display(Image(filename=str(FIGURES_DIR / "fig23_geographic_corridor_risk_matrix.png")))


## 9. Seller Operational Context Analysis
* **Business Question:** Are review ratings driven by persistent seller-level operational quality or by random order-level logistics shocks?
* **Method:** Aggregating seller performance with volume thresholds ($N \ge 100$ and $N \ge 50$) and testing Spearman rank correlations.
* **Findings:** Seller late delivery rate exhibits a significant negative rank correlation with seller average review score ($r_s = -0.42, p < 0.0001$).


In [ ]:
seller_df = pd.read_csv(TABLES_DIR / "seller_context_analysis.csv")
display(seller_df)


## 10. Product Category Confounding & Moderation
* **Business Question:** Do customer expectations vary significantly across product categories during logistics delays?
* **Method:** Evaluating confounding across the top 15 categories and re-confirming interaction effect size ($\eta_p^2$).
* **Findings:** Two-way ANOVA confirms that the interaction term between category and lateness has an effect size of **$\eta_p^2 = 0.072\%$**, while lateness alone explains **$13.16\%$** of variance. Lateness dominates across all categories.


In [ ]:
cat_df = pd.read_csv(TABLES_DIR / "category_control_analysis.csv")
display(cat_df.head(10))


## 11. Freight Burden: Direct vs. Indirect Associations
* **Business Question:** Does expensive shipping directly cause customer dissatisfaction?
* **Method:** Nested logistic regressions controlling for spatial distance, order GMV, and delivery delay.
* **Findings:** Bivariate correlation is weak ($r = -0.065$). In the fully controlled model, freight share exhibits an **Odds Ratio of $0.998$ ($p = 0.42$)**, indicating no statistically significant direct penalty when orders arrive on time.


In [ ]:
freight_df = pd.read_csv(TABLES_DIR / "freight_adjusted_analysis.csv")
display(freight_df)


## 12. Black Friday 2017 Logistics Shock Diagnostic
* **Business Question:** Was the November 2017 rating collapse caused by merchant warehouse backlogs or postal carrier network seizure?
* **Method:** Event-study decomposition comparing Oct 2017 (Baseline) vs Nov 2017 (Surge) vs Dec 2017 (Delivery) vs Jan 2018 (Recovery).
* **Findings:** In November 2017, merchant handling increased by only **$+0.6	ext{ days}$** ($2.7	ext{d} 	o 3.3	ext{d}$), whereas carrier transit surged by **$+5.2	ext{ days}$** ($11.8	ext{d} 	o 17.0	ext{d}$), driving late deliveries from $6.8\%$ to $16.2\%$ and crashing average ratings to $3.82\star$.


In [ ]:
bf_df = pd.read_csv(TABLES_DIR / "black_friday_diagnostic.csv")
display(bf_df)

display(Image(filename=str(FIGURES_DIR / "fig24_black_friday_capacity_shock_decomposition.png")))


## 13. Model Robustness & Sensitivity Checks
To verify that our empirical conclusions are not artifacts of specific modeling choices, we execute 4 sensitivity checks:
1. Alternative outcome: 1-Star ratings only (`review_score == 1`).
2. Alternative outcome: Mild dissatisfaction (`review_score <= 3`).
3. Outlier trimming: Excluding orders with $|	ext{delay}| > 45	ext{ days}$.
4. Quadratic functional form: Including $	ext{delay}^2$.


In [ ]:
rob_df = pd.read_csv(TABLES_DIR / "model_robustness.csv")
display(rob_df)


## 14. Effect Size & Practical Significance Matrix
Every statistical finding is audited to distinguish large-sample statistical significance ($p < 0.001$) from practical operational impact.
* **Delivery Delay ($4–7	ext{d}$ Late):** $	ext{OR} = 9.8	ext{x}$ (Massive practical impact).
* **Survey Timing Asynchrony:** $	ext{OR} = 4.43	ext{x}$ ($26.1\%$ of all low reviews).
* **Carrier vs. Seller Delay:** Carrier $	ext{OR} = 1.48$ vs. Seller $	ext{OR} = 1.12$ per SD.
* **Category Interaction:** $\eta_p^2 = 0.072\%$ (Negligible practical impact).
* **Freight Price Direct Effect:** $	ext{OR} = 0.998$ (Negligible practical impact).


## 15. Formal Module 4 Finding Register
Every finding is classified by evidence, effect size, confidence interval, business impact, and explicit observational causal status.


In [ ]:
find_reg = pd.read_csv(TABLES_DIR / "module_4_finding_register.csv")
display(find_reg)


## 16. Competition Narrative Synthesis
We evaluate three competing narratives against our empirical evidence:
* **Narrative A (Delivery Delay Alone):** Strong evidence, but misses structural geography and feedback loop mechanisms.
* **Narrative B (Geographic Logistics Frictions):** Accurate structural context, but geography operates indirectly through transit time.
* **Narrative C (The Expectation-Fulfillment Gap & Feedback Asynchrony):** **SELECTED WINNING NARRATIVE.** Integrates structural supply concentration in São Paulo ($70.3\%$), conservative promise buffering ($11.95	ext{d}$), non-linear satisfaction collapse beyond Day 3, and automated survey timing asynchrony multiplying dissatisfaction ($4.43	ext{x}$ OR).


## 17. Module Summary & Next Analytical Priorities
Module 4 has delivered:
1. **14 Audited Statistical Tables** in `outputs/tables/`.
2. **6 High-Value Publication Visualizations** in `outputs/figures/`.
3. **10 Empirical Inferences** verified with effect sizes and VIF $< 5.0$.
4. **Automated Test Suite (`tests/test_module4_statistics.py`):** 27 of 27 tests passing (100%).

### Ready for Final Deliverables:
The project now possesses hardened, peer-defensible statistical assets to drive the final executive recommendations, strategy presentation, and competition submission.
